In [8]:
import os
import re
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype, is_integer_dtype, is_float_dtype, is_bool_dtype

file_path = "task-3-Womens Clothing E-Commerce Reviews.csv"

df = pd.read_csv(file_path)

df_original = df.copy()

print("Dataset successfully loaded!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset successfully loaded!
Number of rows: 23486
Number of columns: 11


In [10]:
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [11]:
print("Original column names:")

for column in df.columns:
    print(column)

Original column names:
Unnamed: 0
Clothing ID
Age
Title
Review Text
Rating
Recommended IND
Positive Feedback Count
Division Name
Department Name
Class Name


In [12]:
print(df.dtypes)

Unnamed: 0                 int64
Clothing ID                int64
Age                        int64
Title                        str
Review Text                  str
Rating                     int64
Recommended IND            int64
Positive Feedback Count    int64
Division Name                str
Department Name              str
Class Name                   str
dtype: object


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Unnamed: 0               23486 non-null  int64
 1   Clothing ID              23486 non-null  int64
 2   Age                      23486 non-null  int64
 3   Title                    19676 non-null  str  
 4   Review Text              22641 non-null  str  
 5   Rating                   23486 non-null  int64
 6   Recommended IND          23486 non-null  int64
 7   Positive Feedback Count  23486 non-null  int64
 8   Division Name            23472 non-null  str  
 9   Department Name          23472 non-null  str  
 10  Class Name               23472 non-null  str  
dtypes: int64(6), str(5)
memory usage: 9.5 MB


In [14]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Unnamed: 0                    0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64


In [15]:
def standardize_column_name(column_name):

    column_name = str(column_name).strip()

    column_name = column_name.lower()

    column_name = re.sub(r"[^a-zA-Z0-9]+", "_", column_name)

    column_name = re.sub(r"_+", "_", column_name)

    column_name = column_name.strip("_")

    return column_name

In [16]:
old_columns = df.columns.tolist()

df.columns = [
    standardize_column_name(column)
    for column in df.columns
]

if "unnamed_0" in df.columns:
    df.rename(columns={"unnamed_0": "record_id"}, inplace=True)

new_columns = df.columns.tolist()

In [17]:
column_name_log = pd.DataFrame({
    "original_column_name": old_columns,
    "standardized_column_name": new_columns
})

column_name_log

,original_column_name,standardized_column_name
0,Unnamed: 0,record_id
1,Clothing ID,clothing_id
2,Age,age
3,Title,title
4,Review Text,review_text
5,Rating,rating
6,Recommended IND,recommended_ind
7,Positive Feedback Count,positive_feedback_count
8,Division Name,division_name
9,Department Name,department_name


In [18]:
duplicate_columns = df.columns[df.columns.duplicated()].tolist()

if len(duplicate_columns) == 0:
    print("No duplicate column names found.")
else:
    print("Duplicate column names found:", duplicate_columns)

No duplicate column names found.


In [19]:
print("Standardized column names:")

for column in df.columns:
    print(column)

Standardized column names:
record_id
clothing_id
age
title
review_text
rating
recommended_ind
positive_feedback_count
division_name
department_name
class_name


In [20]:
missing_before = df.isna().sum()

print("Missing values before normalization:")
print(missing_before)

Missing values before normalization:
record_id                     0
clothing_id                   0
age                           0
title                      3810
review_text                 845
rating                        0
recommended_ind               0
positive_feedback_count       0
division_name                14
department_name              14
class_name                   14
dtype: int64


In [21]:
missing_tokens = {
    "",
    " ",
    "na",
    "n/a",
    "nan",
    "null",
    "none",
    "missing",
    "not available",
    "-",
    "--",
    "?"
}

In [22]:
def normalize_missing_values(dataframe):

    cleaned_df = dataframe.copy()

    text_columns = cleaned_df.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:

        cleaned_df[column] = cleaned_df[column].apply(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

        cleaned_df[column] = cleaned_df[column].apply(
            lambda value: pd.NA
            if isinstance(value, str)
            and value.lower() in missing_tokens
            else value
        )

    return cleaned_df

In [23]:
df = normalize_missing_values(df)

print("Missing-value normalization completed.")

Missing-value normalization completed.


In [24]:
missing_after = df.isna().sum()

print("Missing values after normalization:")
print(missing_after)

Missing values after normalization:
record_id                     0
clothing_id                   0
age                           0
title                      3810
review_text                 845
rating                        0
recommended_ind               0
positive_feedback_count       0
division_name                14
department_name              14
class_name                   14
dtype: int64


In [25]:
missing_value_log = pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": missing_after
})

missing_value_log["new_missing_detected"] = (
    missing_value_log["missing_after"]
    - missing_value_log["missing_before"]
)

missing_value_log

,missing_before,missing_after,new_missing_detected
record_id,0,0,0
clothing_id,0,0,0
age,0,0,0
title,3810,3810,0
review_text,845,845,0
rating,0,0,0
recommended_ind,0,0,0
positive_feedback_count,0,0,0
division_name,14,14,0
department_name,14,14,0


In [26]:
rows_before = len(df_original)
rows_after = len(df)

if rows_before == rows_after:
    print("Validation passed: No rows were deleted.")
else:
    print("Warning: Row count has changed.")

Validation passed: No rows were deleted.


In [28]:
def detect_target_type(series, threshold=0.95):

    non_null = series.dropna()

    if len(non_null) == 0:
        return "string", 0.0, "Column contains only missing values"

    if is_bool_dtype(series):
        return "boolean", 1.0, "Already boolean"

    if is_numeric_dtype(series):

        unique_values = set(non_null.unique())

        if unique_values.issubset({0, 1}):
            return "boolean", 1.0, "Only 0 and 1 values detected"

        if is_integer_dtype(series):
            return "integer", 1.0, "Already integer"

        if is_float_dtype(series):
            integer_ratio = (
                non_null.apply(float.is_integer).mean()
            )

            if integer_ratio >= threshold:
                return "integer", integer_ratio, "Whole numbers detected"

            return "float", 1.0, "Decimal numbers detected"

    text_values = non_null.astype(str).str.strip()

    boolean_tokens = {
        "true", "false",
        "yes", "no",
        "y", "n",
        "0", "1"
    }

    boolean_ratio = (
        text_values.str.lower().isin(boolean_tokens).mean()
    )

    if boolean_ratio >= threshold:
        return "boolean", boolean_ratio, "Boolean-like text detected"

    numeric_values = pd.to_numeric(
        text_values.str.replace(",", "", regex=False),
        errors="coerce"
    )

    numeric_ratio = numeric_values.notna().mean()

    if numeric_ratio >= threshold:

        valid_numeric = numeric_values.dropna()

        integer_ratio = (
            valid_numeric % 1 == 0
        ).mean()

        if integer_ratio >= threshold:
            return "integer", numeric_ratio, "Numeric integers detected"

        return "float", numeric_ratio, "Decimal numbers detected"

    date_pattern = (
        r"^\d{1,4}[-/]\d{1,2}[-/]\d{1,4}$"
    )

    date_pattern_ratio = text_values.str.match(
        date_pattern,
        na=False
    ).mean()

    if date_pattern_ratio >= threshold:

        parsed_dates = pd.to_datetime(
            text_values,
            errors="coerce"
        )

        date_ratio = parsed_dates.notna().mean()

        if date_ratio >= threshold:
            return "datetime", date_ratio, "Date values detected"

    unique_count = non_null.nunique()
    unique_ratio = unique_count / len(non_null)

    if unique_count <= 100 and unique_ratio <= 0.20:
        return "category", 1.0 - unique_ratio, "Repeated categories detected"

    return "string", 1.0, "Free text or unique text detected"

In [29]:
type_detection_results = []

for column in df.columns:

    detected_type, confidence, reason = detect_target_type(
        df[column],
        threshold=0.95
    )

    type_detection_results.append({
        "column_name": column,
        "current_dtype": str(df[column].dtype),
        "detected_type": detected_type,
        "confidence": round(confidence * 100, 2),
        "reason": reason
    })

In [30]:
type_detection_report = pd.DataFrame(
    type_detection_results
)

type_detection_report

,column_name,current_dtype,detected_type,confidence,reason
0,record_id,int64,integer,100.00,Already integer
1,clothing_id,int64,integer,100.00,Already integer
2,age,int64,integer,100.00,Already integer
3,title,str,string,100.00,Free text or unique text detected
4,review_text,str,string,100.00,Free text or unique text detected
5,rating,int64,integer,100.00,Already integer
6,recommended_ind,int64,boolean,100.00,Only 0 and 1 values detected
7,positive_feedback_count,int64,integer,100.00,Already integer
8,division_name,str,category,99.99,Repeated categories detected
9,department_name,str,category,99.97,Repeated categories detected


In [31]:
conversion_plan = type_detection_report[
    type_detection_report["current_dtype"]
    != type_detection_report["detected_type"]
]

conversion_plan

,column_name,current_dtype,detected_type,confidence,reason
0,record_id,int64,integer,100.00,Already integer
1,clothing_id,int64,integer,100.00,Already integer
2,age,int64,integer,100.00,Already integer
3,title,str,string,100.00,Free text or unique text detected
4,review_text,str,string,100.00,Free text or unique text detected
5,rating,int64,integer,100.00,Already integer
6,recommended_ind,int64,boolean,100.00,Only 0 and 1 values detected
7,positive_feedback_count,int64,integer,100.00,Already integer
8,division_name,str,category,99.99,Repeated categories detected
9,department_name,str,category,99.97,Repeated categories detected


In [32]:
df_before_type_conversion = df.copy()

print("Backup created successfully.")

Backup created successfully.


In [33]:
def convert_to_boolean(series):

    boolean_mapping = {
        "1": True,
        "0": False,
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
        "y": True,
        "n": False
    }

    def parse_boolean(value):

        if pd.isna(value):
            return pd.NA

        if isinstance(value, bool):
            return value

        value = str(value).strip().lower()

        return boolean_mapping.get(value, pd.NA)

    return series.apply(parse_boolean).astype("boolean")

In [34]:
def clean_numeric_values(series):

    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    cleaned = series.astype("string").str.strip()

    cleaned = cleaned.str.replace(",", "", regex=False)

    cleaned = cleaned.str.replace(
        r"[₹$€£]",
        "",
        regex=True
    )

    return pd.to_numeric(cleaned, errors="coerce")

In [35]:
def convert_column_type(series, target_type):

    if target_type == "integer":

        numeric_series = clean_numeric_values(series)

        return numeric_series.astype("Int64")

    elif target_type == "float":

        numeric_series = clean_numeric_values(series)

        return numeric_series.astype("Float64")

    elif target_type == "boolean":

        return convert_to_boolean(series)

    elif target_type == "datetime":

        return pd.to_datetime(
            series,
            errors="coerce"
        )

    elif target_type == "category":

        return series.astype("category")

    elif target_type == "string":

        return series.astype("string")

    else:
        return series

In [36]:
conversion_log = []

for _, result in type_detection_report.iterrows():

    column = result["column_name"]
    target_type = result["detected_type"]
    confidence = result["confidence"]

    original_series = df[column].copy()
    original_dtype = str(original_series.dtype)

    try:
        converted_series = convert_column_type(
            original_series,
            target_type
        )

        new_invalid_mask = (
            original_series.notna()
            & converted_series.isna()
        )

        invalid_count = new_invalid_mask.sum()

        if invalid_count == 0:

            df[column] = converted_series

            status = "Converted successfully"

        else:

            df[column] = original_series

            status = "Conversion rejected"

        conversion_log.append({
            "column_name": column,
            "original_dtype": original_dtype,
            "target_type": target_type,
            "final_dtype": str(df[column].dtype),
            "confidence_percent": confidence,
            "invalid_values_created": int(invalid_count),
            "status": status
        })

    except Exception as error:

        df[column] = original_series

        conversion_log.append({
            "column_name": column,
            "original_dtype": original_dtype,
            "target_type": target_type,
            "final_dtype": str(df[column].dtype),
            "confidence_percent": confidence,
            "invalid_values_created": 0,
            "status": f"Error: {error}"
        })

In [37]:
conversion_report = pd.DataFrame(conversion_log)

conversion_report

,column_name,original_dtype,target_type,final_dtype,confidence_percent,invalid_values_created,status
0,record_id,int64,integer,Int64,100.00,0,Converted successfully
1,clothing_id,int64,integer,Int64,100.00,0,Converted successfully
2,age,int64,integer,Int64,100.00,0,Converted successfully
3,title,str,string,string,100.00,0,Converted successfully
4,review_text,str,string,string,100.00,0,Converted successfully
5,rating,int64,integer,Int64,100.00,0,Converted successfully
6,recommended_ind,int64,boolean,boolean,100.00,0,Converted successfully
7,positive_feedback_count,int64,integer,Int64,100.00,0,Converted successfully
8,division_name,str,category,category,99.99,0,Converted successfully
9,department_name,str,category,category,99.97,0,Converted successfully


In [38]:
print(df.dtypes)

record_id                     Int64
clothing_id                   Int64
age                           Int64
title                        string
review_text                  string
rating                        Int64
recommended_ind             boolean
positive_feedback_count       Int64
division_name              category
department_name            category
class_name                 category
dtype: object


In [39]:
failed_conversions = conversion_report[
    conversion_report["status"]
    != "Converted successfully"
]

if failed_conversions.empty:
    print("All columns converted safely.")
else:
    print("Some conversions were rejected:")
    display(failed_conversions)

All columns converted safely.


In [40]:
print("Rows before conversion:", len(df_before_type_conversion))
print("Rows after conversion:", len(df))

if len(df_before_type_conversion) == len(df):
    print("Validation passed: No rows were deleted.")
else:
    print("Validation failed: Row count changed.")

Rows before conversion: 23486
Rows after conversion: 23486
Validation passed: No rows were deleted.


In [41]:
validation_rules = {
    "record_id": {
        "minimum": 0,
        "maximum": None
    },
    "clothing_id": {
        "minimum": 0,
        "maximum": None
    },
    "age": {
        "minimum": 13,
        "maximum": 120
    },
    "rating": {
        "minimum": 1,
        "maximum": 5
    },
    "positive_feedback_count": {
        "minimum": 0,
        "maximum": None
    }
}

In [42]:
def validate_numeric_columns(dataframe, rules):

    results = []

    for column, rule in rules.items():

        if column not in dataframe.columns:
            results.append({
                "column_name": column,
                "minimum_allowed": rule["minimum"],
                "maximum_allowed": rule["maximum"],
                "actual_minimum": None,
                "actual_maximum": None,
                "missing_values": None,
                "invalid_values": None,
                "status": "Column not found"
            })

            continue

        series = dataframe[column]

        minimum = rule["minimum"]
        maximum = rule["maximum"]

        invalid_mask = pd.Series(
            False,
            index=series.index
        )

        if minimum is not None:
            invalid_mask = invalid_mask | (
                series.notna() & (series < minimum)
            )

        if maximum is not None:
            invalid_mask = invalid_mask | (
                series.notna() & (series > maximum)
            )

        invalid_count = int(invalid_mask.sum())

        results.append({
            "column_name": column,
            "minimum_allowed": minimum,
            "maximum_allowed": maximum,
            "actual_minimum": series.min(),
            "actual_maximum": series.max(),
            "missing_values": int(series.isna().sum()),
            "invalid_values": invalid_count,
            "status": (
                "Passed"
                if invalid_count == 0
                else "Failed"
            )
        })

    return pd.DataFrame(results)

In [43]:
range_validation_report = validate_numeric_columns(
    df,
    validation_rules
)

range_validation_report

,column_name,minimum_allowed,maximum_allowed,actual_minimum,actual_maximum,missing_values,invalid_values,status
0,record_id,0,NaN,0,23485,0,0,Passed
1,clothing_id,0,NaN,0,1205,0,0,Passed
2,age,13,120.0,18,99,0,0,Passed
3,rating,1,5.0,1,5,0,0,Passed
4,positive_feedback_count,0,NaN,0,122,0,0,Passed


In [44]:
invalid_age_rows = df[
    df["age"].notna()
    & ~df["age"].between(13, 120)
]

invalid_rating_rows = df[
    df["rating"].notna()
    & ~df["rating"].between(1, 5)
]

invalid_feedback_rows = df[
    df["positive_feedback_count"].notna()
    & (df["positive_feedback_count"] < 0)
]

print("Invalid age rows:", len(invalid_age_rows))
print("Invalid rating rows:", len(invalid_rating_rows))
print(
    "Invalid feedback rows:",
    len(invalid_feedback_rows)
)

Invalid age rows: 0
Invalid rating rows: 0
Invalid feedback rows: 0


In [45]:
invalid_recommendation = (
    df["recommended_ind"].notna()
    & ~df["recommended_ind"].isin([True, False])
)

invalid_recommendation_count = int(
    invalid_recommendation.sum()
)

print(
    "Invalid recommendation values:",
    invalid_recommendation_count
)

Invalid recommendation values: 0


In [46]:
duplicate_record_ids = df["record_id"].duplicated().sum()

print("Duplicate record IDs:", duplicate_record_ids)

if duplicate_record_ids == 0:
    print("Record ID validation passed.")
else:
    print("Warning: Duplicate record IDs detected.")

Duplicate record IDs: 0
Record ID validation passed.


In [48]:
category_columns = [
    "division_name",
    "department_name",
    "class_name"
]

for column in category_columns:

    print(f"\nUnique values in {column}:")

    print(
        df[column]
        .dropna()
        .astype("string")
        .sort_values()
        .unique()
    )


Unique values in division_name:
<ArrowStringArray>
['General', 'General Petite', 'Initmates']
Length: 3, dtype: string

Unique values in department_name:
<ArrowStringArray>
['Bottoms', 'Dresses', 'Intimate', 'Jackets', 'Tops', 'Trend']
Length: 6, dtype: string

Unique values in class_name:
<ArrowStringArray>
[       'Blouses', 'Casual bottoms',       'Chemises',        'Dresses',
     'Fine gauge',      'Intimates',        'Jackets',          'Jeans',
          'Knits',       'Layering',        'Legwear',         'Lounge',
      'Outerwear',          'Pants',         'Shorts',         'Skirts',
          'Sleep',       'Sweaters',           'Swim',          'Trend']
Length: 20, dtype: string


In [49]:
category_columns = [
    "division_name",
    "department_name",
    "class_name"
]

In [50]:
def normalize_category_value(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    value = re.sub(r"\s+", " ", value)

    value = value.title()

    return value

In [51]:
category_corrections = {
    "division_name": {
        "Initmates": "Intimates"
    },
    "department_name": {},
    "class_name": {}
}

In [52]:
category_change_records = []

for column in category_columns:

    original_values = df[column].astype("string").copy()

    standardized_values = original_values.apply(
        normalize_category_value
    )

    standardized_values = standardized_values.replace(
        category_corrections.get(column, {})
    )

    changed_mask = (
        original_values.fillna("<MISSING>")
        != standardized_values.fillna("<MISSING>")
    )

    changed_rows = pd.DataFrame({
        "column_name": column,
        "original_value": original_values[changed_mask],
        "standardized_value": standardized_values[changed_mask]
    })

    category_change_records.append(changed_rows)

    df[column] = standardized_values.astype("category")

In [53]:
category_change_log = pd.concat(
    category_change_records,
    ignore_index=True
)

category_change_summary = (
    category_change_log
    .groupby(
        [
            "column_name",
            "original_value",
            "standardized_value"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="rows_changed")
)

category_change_summary

,column_name,original_value,standardized_value,rows_changed
0,class_name,Casual bottoms,Casual Bottoms,2
1,class_name,Fine gauge,Fine Gauge,1100
2,division_name,Initmates,Intimates,1502


In [54]:
for column in category_columns:

    print(f"\nUnique values in {column}:")

    values = (
        df[column]
        .dropna()
        .astype("string")
        .sort_values()
        .unique()
    )

    print(values)


Unique values in division_name:
<ArrowStringArray>
['General', 'General Petite', 'Intimates']
Length: 3, dtype: string

Unique values in department_name:
<ArrowStringArray>
['Bottoms', 'Dresses', 'Intimate', 'Jackets', 'Tops', 'Trend']
Length: 6, dtype: string

Unique values in class_name:
<ArrowStringArray>
[       'Blouses', 'Casual Bottoms',       'Chemises',        'Dresses',
     'Fine Gauge',      'Intimates',        'Jackets',          'Jeans',
          'Knits',       'Layering',        'Legwear',         'Lounge',
      'Outerwear',          'Pants',         'Shorts',         'Skirts',
          'Sleep',       'Sweaters',           'Swim',          'Trend']
Length: 20, dtype: string


In [55]:
print("Missing values after category standardization:")

print(
    df[category_columns]
    .isna()
    .sum()
)

Missing values after category standardization:
division_name      14
department_name    14
class_name         14
dtype: int64


In [56]:
if len(df) == len(df_original):
    print("Validation passed: No rows were deleted.")
else:
    print("Validation failed: Row count changed.")

Validation passed: No rows were deleted.


In [57]:
df_before_missing_handling = df.copy()

missing_before_handling = df.isna().sum()

print("Missing values before handling:")
print(
    missing_before_handling[
        missing_before_handling > 0
    ]
)

Missing values before handling:
title              3810
review_text         845
division_name        14
department_name      14
class_name           14
dtype: int64


In [58]:
df["title_missing"] = df["title"].isna()
df["review_text_missing"] = df["review_text"].isna()

df["title_missing"] = df["title_missing"].astype("boolean")
df["review_text_missing"] = (
    df["review_text_missing"].astype("boolean")
)

In [59]:
df["title"] = (
    df["title"]
    .astype("string")
    .fillna("Not Provided")
)

df["review_text"] = (
    df["review_text"]
    .astype("string")
    .fillna("Not Provided")
)

In [60]:
valid_review_data = df[
    df["review_text_missing"] == False
].copy()

In [61]:
category_columns = [
    "division_name",
    "department_name",
    "class_name"
]

for column in category_columns:

    df[column] = (
        df[column]
        .astype("string")
        .fillna("Unknown")
        .astype("category")
    )

In [62]:
missing_after_handling = df.isna().sum()

missing_handling_log = pd.DataFrame({
    "missing_before": missing_before_handling,
    "missing_after": missing_after_handling
})

missing_handling_log["values_handled"] = (
    missing_handling_log["missing_before"]
    - missing_handling_log["missing_after"]
)

missing_handling_log[
    missing_handling_log["values_handled"] > 0
]

,missing_before,missing_after,values_handled
class_name,14.0,0,14.0
department_name,14.0,0,14.0
division_name,14.0,0,14.0
review_text,845.0,0,845.0
title,3810.0,0,3810.0


In [63]:
remaining_missing = int(df.isna().sum().sum())

print("Total remaining missing values:", remaining_missing)

if remaining_missing == 0:
    print("Missing-value validation passed.")
else:
    print("Some missing values are still present.")

Total remaining missing values: 0
Missing-value validation passed.


In [64]:
print(
    "Originally missing titles:",
    df["title_missing"].sum()
)

print(
    "Originally missing reviews:",
    df["review_text_missing"].sum()
)

Originally missing titles: 3810
Originally missing reviews: 845


In [65]:
print("Original rows:", len(df_original))
print("Current rows:", len(df))

if len(df) == len(df_original):
    print("Validation passed: No rows were deleted.")
else:
    print("Validation failed: Row count changed.")

Original rows: 23486
Current rows: 23486
Validation passed: No rows were deleted.


In [66]:
validation_checks = []

def add_validation(check_name, condition, details):
    validation_checks.append({
        "validation_check": check_name,
        "status": "PASS" if bool(condition) else "FAIL",
        "details": details
    })

In [67]:
row_count_valid = len(df) == len(df_original)

add_validation(
    "Row count preserved",
    row_count_valid,
    f"Original: {len(df_original)}, Cleaned: {len(df)}"
)

In [68]:
standard_column_pattern = r"^[a-z][a-z0-9_]*$"

valid_column_names = all(
    bool(re.match(standard_column_pattern, column))
    for column in df.columns
)

add_validation(
    "Column names standardized",
    valid_column_names,
    "All columns must use lowercase snake_case"
)

In [69]:
duplicate_column_count = int(
    df.columns.duplicated().sum()
)

add_validation(
    "No duplicate column names",
    duplicate_column_count == 0,
    f"Duplicate columns: {duplicate_column_count}"
)

In [70]:
total_missing = int(df.isna().sum().sum())

add_validation(
    "No unhandled missing values",
    total_missing == 0,
    f"Remaining missing values: {total_missing}"
)

In [71]:
duplicate_record_ids = int(
    df["record_id"].duplicated().sum()
)

add_validation(
    "Record IDs are unique",
    duplicate_record_ids == 0,
    f"Duplicate record IDs: {duplicate_record_ids}"
)

In [72]:
age_valid = df["age"].between(13, 120).all()

rating_valid = df["rating"].between(1, 5).all()

feedback_valid = (
    df["positive_feedback_count"] >= 0
).all()

recommendation_valid = (
    df["recommended_ind"]
    .isin([True, False])
    .all()
)

add_validation(
    "Age range valid",
    age_valid,
    "Required range: 13 to 120"
)

add_validation(
    "Rating range valid",
    rating_valid,
    "Required range: 1 to 5"
)

add_validation(
    "Feedback count valid",
    feedback_valid,
    "Feedback count must be zero or positive"
)

add_validation(
    "Recommendation values valid",
    recommendation_valid,
    "Only True and False are allowed"
)

In [73]:
numeric_column_mapping = {
    "Unnamed: 0": "record_id",
    "Clothing ID": "clothing_id",
    "Age": "age",
    "Rating": "rating",
    "Recommended IND": "recommended_ind",
    "Positive Feedback Count": "positive_feedback_count"
}

for original_column, cleaned_column in numeric_column_mapping.items():

    original_values = pd.to_numeric(
        df_original[original_column],
        errors="coerce"
    )

    if cleaned_column == "recommended_ind":
        cleaned_values = (
            df[cleaned_column]
            .astype("Int64")
        )
    else:
        cleaned_values = pd.to_numeric(
            df[cleaned_column],
            errors="coerce"
        )

    values_preserved = (
        original_values.reset_index(drop=True)
        .eq(cleaned_values.reset_index(drop=True))
        .all()
    )

    add_validation(
        f"{cleaned_column} values preserved",
        values_preserved,
        f"Compared with original column: {original_column}"
    )

In [74]:
original_titles = (
    df_original["Title"]
    .dropna()
    .astype("string")
    .str.strip()
)

cleaned_titles = df.loc[
    df_original["Title"].notna(),
    "title"
].astype("string")

titles_preserved = (
    original_titles.reset_index(drop=True)
    .eq(cleaned_titles.reset_index(drop=True))
    .all()
)

add_validation(
    "Original title text preserved",
    titles_preserved,
    "Non-missing title values compared"
)

In [75]:
original_reviews = (
    df_original["Review Text"]
    .dropna()
    .astype("string")
    .str.strip()
)

cleaned_reviews = df.loc[
    df_original["Review Text"].notna(),
    "review_text"
].astype("string")

reviews_preserved = (
    original_reviews.reset_index(drop=True)
    .eq(cleaned_reviews.reset_index(drop=True))
    .all()
)

add_validation(
    "Original review text preserved",
    reviews_preserved,
    "Non-missing review values compared"
)

In [76]:
expected_dtypes = {
    "record_id": "Int64",
    "clothing_id": "Int64",
    "age": "Int64",
    "title": "string",
    "review_text": "string",
    "rating": "Int64",
    "recommended_ind": "boolean",
    "positive_feedback_count": "Int64",
    "division_name": "category",
    "department_name": "category",
    "class_name": "category",
    "title_missing": "boolean",
    "review_text_missing": "boolean"
}

dtype_records = []

for column, expected_dtype in expected_dtypes.items():

    actual_dtype = str(df[column].dtype)

    dtype_records.append({
        "column_name": column,
        "expected_dtype": expected_dtype,
        "actual_dtype": actual_dtype,
        "status": (
            "PASS"
            if actual_dtype == expected_dtype
            else "FAIL"
        )
    })

dtype_validation_report = pd.DataFrame(dtype_records)

dtype_validation_report

,column_name,expected_dtype,actual_dtype,status
0,record_id,Int64,Int64,PASS
1,clothing_id,Int64,Int64,PASS
2,age,Int64,Int64,PASS
3,title,string,string,PASS
4,review_text,string,string,PASS
5,rating,Int64,Int64,PASS
6,recommended_ind,boolean,boolean,PASS
7,positive_feedback_count,Int64,Int64,PASS
8,division_name,category,category,PASS
9,department_name,category,category,PASS


In [77]:
final_validation_report = pd.DataFrame(
    validation_checks
)

final_validation_report

,validation_check,status,details
0,Row count preserved,PASS,"Original: 23486, Cleaned: 23486"
1,Column names standardized,PASS,All columns must use lowercase snake_case
2,No duplicate column names,PASS,Duplicate columns: 0
3,No unhandled missing values,PASS,Remaining missing values: 0
4,Record IDs are unique,PASS,Duplicate record IDs: 0
5,Age range valid,PASS,Required range: 13 to 120
6,Rating range valid,PASS,Required range: 1 to 5
7,Feedback count valid,PASS,Feedback count must be zero or positive
8,Recommendation values valid,PASS,Only True and False are allowed
9,record_id values preserved,PASS,Compared with original column: Unnamed: 0


In [78]:
all_checks_passed = (
    final_validation_report["status"] == "PASS"
).all()

all_dtypes_passed = (
    dtype_validation_report["status"] == "PASS"
).all()

if all_checks_passed and all_dtypes_passed:
    print("FINAL RESULT: All validation checks passed.")
    print("Dataset is clean, consistent and ready for analysis.")
else:
    print("FINAL RESULT: Some validation checks failed.")

    print("\nFailed validation checks:")
    display(
        final_validation_report[
            final_validation_report["status"] == "FAIL"
        ]
    )

    print("\nFailed data-type checks:")
    display(
        dtype_validation_report[
            dtype_validation_report["status"] == "FAIL"
        ]
    )

FINAL RESULT: All validation checks passed.
Dataset is clean, consistent and ready for analysis.


In [79]:
output_folder = "womens_clothing_project_output"

os.makedirs(
    output_folder,
    exist_ok=True
)

print("Output folder created:", output_folder)

Output folder created: womens_clothing_project_output


In [80]:
if not all_checks_passed:
    raise ValueError(
        "Dataset export stopped because validation checks failed."
    )

if not all_dtypes_passed:
    raise ValueError(
        "Dataset export stopped because data-type checks failed."
    )

print("All validations passed. Export is allowed.")

All validations passed. Export is allowed.


In [81]:
cleaned_file_path = os.path.join(
    output_folder,
    "womens_clothing_cleaned.csv"
)

df.to_csv(
    cleaned_file_path,
    index=False
)

print("Cleaned dataset saved:")
print(cleaned_file_path)

Cleaned dataset saved:
womens_clothing_project_output\womens_clothing_cleaned.csv


In [82]:
column_name_log.to_csv(
    os.path.join(
        output_folder,
        "column_name_log.csv"
    ),
    index=False
)

In [83]:
type_detection_report.to_csv(
    os.path.join(
        output_folder,
        "type_detection_report.csv"
    ),
    index=False
)

In [84]:
conversion_report.to_csv(
    os.path.join(
        output_folder,
        "type_conversion_report.csv"
    ),
    index=False
)

In [85]:
category_change_summary.to_csv(
    os.path.join(
        output_folder,
        "category_change_log.csv"
    ),
    index=False
)

In [86]:
missing_handling_log.to_csv(
    os.path.join(
        output_folder,
        "missing_value_log.csv"
    )
)

In [87]:
range_validation_report.to_csv(
    os.path.join(
        output_folder,
        "range_validation_report.csv"
    ),
    index=False
)

final_validation_report.to_csv(
    os.path.join(
        output_folder,
        "final_validation_report.csv"
    ),
    index=False
)

dtype_validation_report.to_csv(
    os.path.join(
        output_folder,
        "dtype_validation_report.csv"
    ),
    index=False
)

In [88]:
final_schema = pd.DataFrame({
    "column_name": df.columns,
    "data_type": [
        str(df[column].dtype)
        for column in df.columns
    ],
    "missing_values": [
        int(df[column].isna().sum())
        for column in df.columns
    ],
    "unique_values": [
        int(df[column].nunique(dropna=True))
        for column in df.columns
    ]
})

final_schema.to_csv(
    os.path.join(
        output_folder,
        "final_data_schema.csv"
    ),
    index=False
)

final_schema

,column_name,data_type,missing_values,unique_values
0,record_id,Int64,0,23486
1,clothing_id,Int64,0,1206
2,age,Int64,0,77
3,title,string,0,13994
4,review_text,string,0,22635
5,rating,Int64,0,5
6,recommended_ind,boolean,0,2
7,positive_feedback_count,Int64,0,82
8,division_name,category,0,4
9,department_name,category,0,7


In [89]:
saved_df = pd.read_csv(cleaned_file_path)

print("Saved rows:", saved_df.shape[0])
print("Saved columns:", saved_df.shape[1])
print(
    "Missing values in saved file:",
    saved_df.isna().sum().sum()
)

Saved rows: 23486
Saved columns: 13
Missing values in saved file: 0


In [90]:
print("Generated project files:\n")

for file_name in sorted(os.listdir(output_folder)):
    print(file_name)

Generated project files:

category_change_log.csv
column_name_log.csv
dtype_validation_report.csv
final_data_schema.csv
final_validation_report.csv
missing_value_log.csv
range_validation_report.csv
type_conversion_report.csv
type_detection_report.csv
womens_clothing_cleaned.csv
